This notebook loads in the CIFAR-10 database of images, trains a CNN on the dataset, and exports the model weights

In [25]:
import torch
import torchvision
from torchvision.transforms import v2

device = torch.device(torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

mean_list = (0.4914, 0.4822, 0.4465)
std_list = (0.247, 0.2435, 0.2616)

# Transforms to turn images into 3D tensors
# Random cropping and horizontal flipping to introduce variability into training data
train_transform = v2.Compose([
    v2.RandomCrop(32, padding=4),
    v2.RandomHorizontalFlip(),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean_list, std_list)
])

val_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean_list, std_list)
])

# Fixed random seed
seeded_generator = torch.Generator().manual_seed(42)
perm = torch.randperm(50000, generator=seeded_generator)

# Selects 45000 images from the dataset for training, 5000 for validation
train_indices = perm[:45000]
val_indices = perm[45000:]

batch_size = 128

# Defines the training, eval, and testing data sets and data loaders
train_full_aug = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
train_full_plain = torchvision.datasets.CIFAR10(root="./data", train=True, download=False, transform=val_transform)

trainset = torch.utils.data.Subset(train_full_aug, train_indices)
valset = torch.utils.data.Subset(train_full_plain, val_indices)
testset = torchvision.datasets.CIFAR10(root="./data", train=False, download=False, transform=val_transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader = torch.utils.data.DataLoader(evalset, batch_size=batch_size * 2, shuffle=False, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size * 2, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')


The above code imports necessary libraries and creates a transform that turns images into 3D tensors for processing. It then creates the training, validation, and testing sets alongside the data loaders, and is followed by defining the classes by which the model should classify images.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# Class for the CNN architecture
class Net(nn.Module):
  def __init__(self):
    super().__init__()
    self.pool = nn.MaxPool2d(2, 2)
    self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
    self.conv2 = nn.Conv2d(32, 32, 3, padding=1)
    self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
    self.conv4 = nn.Conv2d(64, 64, 3, padding=1)
    self.conv5 = nn.Conv2d(64, 128, 3, padding=1)
    self.conv6 = nn.Conv2d(128, 128, 3, padding=1)

    self.fc1 = nn.Linear(128 * 4 * 4, 256)
    self.fc2 = nn.Linear(256, 10)
    self.dropout = nn.Dropout(p=0.5)

    self.bn1 = nn.BatchNorm2d(32)
    self.bn2 = nn.BatchNorm2d(32)
    self.bn3 = nn.BatchNorm2d(64)
    self.bn4 = nn.BatchNorm2d(64)
    self.bn5 = nn.BatchNorm2d(128)
    self.bn6 = nn.BatchNorm2d(128)

  def forward(self, x):
    # Applies 6 kernel convolutions to scale channels from 3 to 128
    # Applies 3 pooling transformations to scale images from 32x32 to 4x4
    x = F.relu(self.bn1(self.conv1(x)))
    x = F.relu(self.bn2(self.conv2(x)))
    x = self.pool(x)
    x = F.relu(self.bn3(self.conv3(x)))
    x = F.relu(self.bn4(self.conv4(x)))
    x = self.pool(x)
    x = F.relu(self.bn5(self.conv5(x)))
    x = F.relu(self.bn6(self.conv6(x)))
    x = self.pool(x)
    # Flattens input and performs linear transformations with dropout before final step
    x = torch.flatten(x, 1)
    x = F.relu(self.fc1(x))
    x = self.dropout(x)
    x = self.fc2(x)
    return x

net = Net()
net.to(device)

Net(
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv6): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=2048, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=10, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn4): Bat

The above cell defines the CNN model architecture, starting with kernel convolutions and pooling to scale channels and size of images, and then flattening the images before applying linear transformations and a dropout on them.

In [ ]:
# Function to evaluate model performance on a dataset
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            predicted = outputs.argmax(dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return correct / total

In [ ]:
import torch.optim as optim

# Sets loss function, optimization function, and schedules optimization
EPOCHS = 30

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr = 0.05, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer=optimizer, max_lr=0.1, epochs=EPOCHS, steps_per_epoch=len(trainloader))

best_val_acc = 0.0

# Loops for all epochs and trains the model using images in the training dataset
for epoch in range(EPOCHS):
  net.train()
  running_loss = 0.0
  batches = 0
  for i, data in enumerate(trainloader, 0):
    inputs, labels = data[0].to(device), data[1].to(device)

    optimizer.zero_grad()

    outputs = net(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    scheduler.step()

    running_loss += loss.item()
    batches += 1

  # Records best performing model weights
  val_acc = evaluate(net, valloader, device)
  if (val_acc > best_val_acc):
    best_val_acc = val_acc
    torch.save(net.state_dict(), "cifar_net_best.pt")

  average_loss = running_loss / batches
  print("Epoch: ", str(epoch + 1) + ", average loss: " + str(average_loss) + ", accuracy: " + str(val_acc))

print("Finished Training")

Epoch:  1, average loss: 1.6079190993173555, accuracy: 0.5552
Epoch:  2, average loss: 1.2672312777828088, accuracy: 0.5816
Epoch:  3, average loss: 1.1376719505272128, accuracy: 0.6666
Epoch:  4, average loss: 0.9882524093105034, accuracy: 0.6714
Epoch:  5, average loss: 0.8342593606900085, accuracy: 0.6668
Epoch:  6, average loss: 0.7536457655104724, accuracy: 0.717
Epoch:  7, average loss: 0.6930749981071461, accuracy: 0.7572
Epoch:  8, average loss: 0.6492375060915947, accuracy: 0.7524
Epoch:  9, average loss: 0.6209700529209592, accuracy: 0.7978
Epoch:  10, average loss: 0.6023549473082478, accuracy: 0.7638
Epoch:  11, average loss: 0.583878836594522, accuracy: 0.757
Epoch:  12, average loss: 0.5603369565003298, accuracy: 0.8096
Epoch:  13, average loss: 0.5404450865462422, accuracy: 0.7184
Epoch:  14, average loss: 0.5332613837143237, accuracy: 0.776
Epoch:  15, average loss: 0.5146659269336272, accuracy: 0.7672
Epoch:  16, average loss: 0.5062576967545531, accuracy: 0.808
Epoch:

The above code trains the model using gradient descent and saving model weights of the epoch with the best validation accuracy

In [30]:
# Tests accuracy of CNN on the testing dataset
testNet = Net()
testNet.to(device)
testNet.load_state_dict(torch.load("./cifar_net_best.pt", weights_only=True))
print("Accuracy = " + str(evaluate(testNet, testloader, device)))

Accuracy = 0.8983


In [32]:
import json
import numpy as np
import torch
import torchvision

# Raw test set, x and y are inputs and outputs
raw_test = torchvision.datasets.CIFAR10(root="./data", train=False, download=False)

test_x = raw_test.data
test_y = np.array(raw_test.targets, dtype=np.int64)

print("test_x:", test_x.shape, test_x.dtype)
print("test_y:", test_y.shape, test_y.dtype)
print("pixel range:", test_x.min(), "to", test_x.max())

np.save("test_x.npy", test_x)
np.save("test_y.npy", test_y)

# Model Statistics
with open("cifar10_stats.json", "w") as f:
    json.dump({"mean": list(mean_list), "std": list(std_list)}, f)

# Replicates what the Jetson will do
x = np.load("test_x.npy").astype(np.float32) / 255.0
x = (x - np.array(mean_list, dtype=np.float32)) / np.array(std_list, dtype=np.float32)
x = np.ascontiguousarray(x.transpose(0, 3, 1, 2))
y = np.load("test_y.npy")

testNet.eval()
correct = 0
with torch.no_grad():
    for i in range(0, len(x), 500):
        xb = torch.from_numpy(x[i:i + 500]).to(device)
        yb = torch.from_numpy(y[i:i + 500]).to(device)
        correct += (testNet(xb).argmax(dim=1) == yb).sum().item()

print("Accuracy from raw arrays:", correct / len(x))

test_x: (10000, 32, 32, 3) uint8
test_y: (10000,) int64
pixel range: 0 to 255
Accuracy from raw arrays: 0.8983


In [34]:
from google.colab import files

files.download("cifar_net_best.pt")
files.download("test_x.npy")
files.download("test_y.npy")
files.download("cifar10_stats.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The above code exports the model weights, model statistics, and testing dataset inputs and outputs to be used for testing on the Jetson Nano